# Stage 4 — Translate-Then-Cluster

Translates non-English text to English, applies BERTopic to the translations, and maps the resulting topics back to the original-language source text. Evaluates topic homogeneity across three parallel setups:

1. **Arabic content** — cluster original Arabic text using a multilingual sentence model (baseline)
2. **English gold standard** — cluster human-translated English (upper bound)
3. **English translation** — cluster MBART-translated English (the real scenario)

**Runtime:** Google Colab + Google Drive  
**Storage:** All inputs and outputs read from / written to a project folder on Google Drive.  
**Library:** [`multilingual-topic-modeling`](https://github.com/ay94/multilingual-topic-modeling)

**Outputs:**
- Three CSVs: `topics_translation.csv`, `topics_gold.csv`, `topics_arabic.csv`
- Entropy table per setup for homogeneity assessment
- Annotation sample (top 10 + bottom 5 topics, 10% per topic)

See [`topic-modeling-recipes/docs/translation/`](https://github.com/ay94/topic-modeling-recipes/tree/main/docs/translation) for the full methodology and findings.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%%capture
!pip install multilingual-topic-modeling transformers sentencepiece accelerate datasets bertopic umap-learn hdbscan --quiet
import nltk
nltk.download('wordnet')
nltk.download('punkt_tab')

In [ ]:
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm

from multilingual_topic import FileHandler, Translator

from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Configuration

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
DRIVE_FOLDER = '/content/drive/MyDrive/YOUR_PROJECT/Topic Modelling Workflow'

SAMPLE_SIZE      = 10_000   # UN parallel corpus rows to use
TRANSLATION_BATCH = 8       # MBART batch size — reduce if OOM
MAX_LENGTH       = 512

# UMAP / HDBSCAN — identical across all three setups
UMAP_COMPONENTS   = 5
UMAP_NEIGHBORS    = 15
HDBSCAN_MIN_SIZE  = 30
HDBSCAN_MIN_SAMPLES = 10

# Annotation sample
TOP_N_TOPICS    = 10   # largest topics
BOTTOM_N_TOPICS = 5    # smallest topics
SAMPLE_FRACTION = 0.10 # 10% per topic

# Homogeneity threshold: normalised entropy above this → heterogeneous
ENTROPY_THRESHOLD = 0.5
# ──────────────────────────────────────────────────────────────────────────────

fh = FileHandler(DRIVE_FOLDER)

## 1. Load dataset

UN Parallel Corpus (`un_pc`) — manually translated UN documents 1990–2014.  
Arabic–English split: each row has an Arabic source and a human-translated English gold standard.

In [ ]:
from datasets import load_dataset

raw = load_dataset('un_pc', 'ar-en', split=f'train[:{SAMPLE_SIZE}]', trust_remote_code=True)
df = pd.DataFrame({
    'arabic': [r['translation']['ar'] for r in raw],
    'english_gold': [r['translation']['en'] for r in raw],
})
df = df.dropna().reset_index(drop=True)
print(f'{len(df):,} sentence pairs loaded')
df.head(3)

## 2. Translate Arabic → English

Uses `Translator` from `multilingual_topic`. Wraps `facebook/mbart-large-50-many-to-many-mmt`  
with DataLoader-based batching and automatic GPU detection.

In [ ]:
translator = Translator(src_lang='ar', tgt_lang='en', device=DEVICE)
print(f'Translator loaded — model: {translator._model_name}')

In [ ]:
translations = translator.translate(
    list(df['arabic']),
    batch_size=TRANSLATION_BATCH,
    max_length=MAX_LENGTH,
    show_progress=True,
)
df['english_translation'] = translations
print('Translation complete.')

# Save checkpoint — translation is the expensive step
fh.save(df, 'outputs/un_pc_translated.csv')
df.head(3)

In [ ]:
# Reload checkpoint if translation already done
# df = pd.read_csv(fh.create_filename('outputs/un_pc_translated.csv'))

## 3. Compute embeddings

Three sentence models — one per setup:

| Setup | Model | Rationale |
|---|---|---|
| Arabic content | `paraphrase-multilingual-MiniLM-L12-v2` | Multilingual, supports Arabic |
| English gold | `all-MiniLM-L6-v2` | English-only, fast |
| English translation | `all-MiniLM-L6-v2` | Same as gold for fair comparison |

In [ ]:
multilingual_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=DEVICE)
english_model      = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

print('Encoding Arabic...')
emb_arabic = multilingual_model.encode(list(df['arabic']), batch_size=64, show_progress_bar=True)

print('Encoding English gold...')
emb_gold = english_model.encode(list(df['english_gold']), batch_size=64, show_progress_bar=True)

print('Encoding English translation...')
emb_translation = english_model.encode(list(df['english_translation']), batch_size=64, show_progress_bar=True)

print(f'Embedding shapes: Arabic={emb_arabic.shape}, Gold={emb_gold.shape}, Translation={emb_translation.shape}')

## 4. Topic modelling — three setups

Identical UMAP and HDBSCAN parameters across all three setups.  
The only variable is the embedding (and therefore the content the model sees).

In [ ]:
def build_topic_model():
    umap_model = UMAP(
        n_components=UMAP_COMPONENTS,
        n_neighbors=UMAP_NEIGHBORS,
        metric='cosine',
        random_state=42,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        prediction_data=True,
    )
    vectorizer = CountVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))
    return BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer,
        verbose=True,
    )

In [ ]:
docs_translation = list(df['english_translation'])
docs_gold        = list(df['english_gold'])
docs_arabic      = list(df['arabic'])

print('--- Setup 1: English translation ---')
model_translation = build_topic_model()
topics_translation, _ = model_translation.fit_transform(docs_translation, embeddings=emb_translation)
df['topic_translation'] = topics_translation
print(f'Topics found: {model_translation.get_topic_info().shape[0] - 1}')

In [ ]:
print('--- Setup 2: English gold standard ---')
model_gold = build_topic_model()
topics_gold, _ = model_gold.fit_transform(docs_gold, embeddings=emb_gold)
df['topic_gold'] = topics_gold
print(f'Topics found: {model_gold.get_topic_info().shape[0] - 1}')

In [ ]:
print('--- Setup 3: Arabic content ---')
model_arabic = build_topic_model()
topics_arabic, _ = model_arabic.fit_transform(docs_arabic, embeddings=emb_arabic)
df['topic_arabic'] = topics_arabic
print(f'Topics found: {model_arabic.get_topic_info().shape[0] - 1}')

In [ ]:
# Summary
print('Topic counts:')
print(f'  Translation:     {model_translation.get_topic_info().shape[0] - 1}')
print(f'  English gold:    {model_gold.get_topic_info().shape[0] - 1}')
print(f'  Arabic content:  {model_arabic.get_topic_info().shape[0] - 1}')

## 5. Save topic assignments

In [ ]:
fh.save(df, 'outputs/topics_all_setups.csv')

# Per-setup outputs — Arabic text + topic assignment for analyst
out_translation = df[['arabic', 'english_translation', 'topic_translation']].copy()
out_gold        = df[['arabic', 'english_gold', 'topic_gold']].copy()
out_arabic      = df[['arabic', 'topic_arabic']].copy()

fh.save(out_translation, 'outputs/topics_translation.csv')
fh.save(out_gold,        'outputs/topics_gold.csv')
fh.save(out_arabic,      'outputs/topics_arabic.csv')

print('Saved.')

## 6. Topic keyword inspection

Review top keywords per topic across each setup.

In [ ]:
def top_keywords(model, n_topics=10):
    info = model.get_topic_info()
    info = info[info['Topic'] != -1].head(n_topics)
    rows = []
    for tid in info['Topic']:
        words = [w for w, _ in model.get_topic(tid)[:8]]
        rows.append({'topic': tid, 'size': info.loc[info['Topic'] == tid, 'Count'].values[0], 'keywords': ', '.join(words)})
    return pd.DataFrame(rows)

print('=== Translation setup — top 10 topics ===')
display(top_keywords(model_translation))

print('=== English gold — top 10 topics ===')
display(top_keywords(model_gold))

print('=== Arabic content — top 10 topics ===')
display(top_keywords(model_arabic))

## 7. Entropy-based homogeneity assessment

After annotation, compute entropy per topic to quantify homogeneity.

**Formula:**  
`H(X) = -sum(p(x) * log2(p(x)))`  
`H_norm = H(X) / log2(n)` where n = number of distinct descriptions

Topics with `H_norm < 0.5` are homogeneous; above that threshold, heterogeneous.

**Input:** annotated CSV with a `description` column added per message.

In [ ]:
def compute_entropy(description_series):
    counts = description_series.value_counts(normalize=True)
    n = len(counts)
    if n == 1:
        return 0.0, 0.0
    h = -np.sum(counts * np.log2(counts))
    h_max = np.log2(n)
    return round(h, 4), round(h / h_max, 4)


def homogeneity_table(annotated_df, topic_col, description_col, threshold=ENTROPY_THRESHOLD):
    rows = []
    for topic_id, group in annotated_df.groupby(topic_col):
        h, h_norm = compute_entropy(group[description_col])
        rows.append({
            'topic': topic_id,
            'n_messages': len(group),
            'n_descriptions': group[description_col].nunique(),
            'entropy': h,
            'entropy_norm': h_norm,
            'homogeneous': h_norm < threshold,
        })
    return pd.DataFrame(rows).sort_values('entropy_norm', ascending=False)

In [ ]:
# Load annotated outputs — add 'description' column per message then re-save
# annotated_translation = pd.read_csv(fh.create_filename('outputs/annotated_translation.csv'))
# annotated_gold        = pd.read_csv(fh.create_filename('outputs/annotated_gold.csv'))
# annotated_arabic      = pd.read_csv(fh.create_filename('outputs/annotated_arabic.csv'))
#
# entropy_translation = homogeneity_table(annotated_translation, 'topic_translation', 'description')
# entropy_gold        = homogeneity_table(annotated_gold,        'topic_gold',        'description')
# entropy_arabic      = homogeneity_table(annotated_arabic,      'topic_arabic',      'description')
#
# print('Translation setup:')
# display(entropy_translation)
# print('English gold:')
# display(entropy_gold)
# print('Arabic content:')
# display(entropy_arabic)

## 8. Annotation sample

Select top 10 and bottom 5 topics from each setup. Sample 10% of messages per topic.  
Save to Google Sheets or CSV for manual annotation.

In [ ]:
def annotation_sample(df, topic_col, text_cols, model, top_n=TOP_N_TOPICS, bottom_n=BOTTOM_N_TOPICS, frac=SAMPLE_FRACTION):
    info = model.get_topic_info()
    info = info[info['Topic'] != -1]
    top_topics    = info.nlargest(top_n, 'Count')['Topic'].tolist()
    bottom_topics = info.nsmallest(bottom_n, 'Count')['Topic'].tolist()
    selected = list(set(top_topics + bottom_topics))

    frames = []
    for tid in selected:
        subset = df[df[topic_col] == tid]
        n = max(1, int(len(subset) * frac))
        sample = subset.sample(n=min(n, len(subset)), random_state=42)
        frames.append(sample)
    result = pd.concat(frames, ignore_index=True)
    result['description'] = ''  # annotator fills this in
    return result[text_cols + [topic_col, 'description']]


sample_translation = annotation_sample(
    df, 'topic_translation',
    text_cols=['arabic', 'english_translation'],
    model=model_translation,
)
sample_gold = annotation_sample(
    df, 'topic_gold',
    text_cols=['arabic', 'english_gold'],
    model=model_gold,
)
sample_arabic = annotation_sample(
    df, 'topic_arabic',
    text_cols=['arabic'],
    model=model_arabic,
)

fh.save(sample_translation, 'outputs/annotation_sample_translation.csv')
fh.save(sample_gold,        'outputs/annotation_sample_gold.csv')
fh.save(sample_arabic,      'outputs/annotation_sample_arabic.csv')

print(f'Annotation samples saved:')
print(f'  Translation: {len(sample_translation):,} messages')
print(f'  Gold:        {len(sample_gold):,} messages')
print(f'  Arabic:      {len(sample_arabic):,} messages')

## 9. Reference results

From the original Arabic→English study on UN Parallel Corpus (10K sentences, MBART, GPU A100):

| Setup | Topics | Heterogeneous topics | % heterogeneous |
|---|---|---|---|
| English translation | 67 | 5 / 15 annotated | 30% |
| English gold standard | 69 | 1 / 15 annotated | 6% |
| Arabic content | 64 | 1 / 15 annotated | 6% |

Shared topic discussions across all three setups: Cost, Human Rights, Law, Resolution/Committee.  
The translation setup produces semantically aligned but less homogeneous topic clusters.

See [`topic-modeling-recipes/docs/translation/`](https://github.com/ay94/topic-modeling-recipes/tree/main/docs/translation) for the full findings.